# Silver — Cleansed & Conformed
Current rows only, derived business columns, referential-integrity filtering into `silver_*`.

In [ ]:
# ============================================================
# NOTEBOOK 2: SILVER  —  cleansed, conformed, integrity-checked
# ============================================================
from pyspark.sql import functions as F

B, SLV = "bronze_", "silver_"
def bronze(t): return spark.table(f"{B}{t}")

# ---------- Dimensions: keep current, tidy ----------
dim_provider = (bronze("dimprovider").where(F.col("_IsCurrent"))
    .withColumn("ProviderFullName", F.concat_ws(", ", "LastName", "FirstName"))
    .withColumn("IsActive", F.col("IsProviderActive").cast("boolean")))
dim_patient = (bronze("dimpatient").where(F.col("_IsCurrent"))
    .withColumn("Age", F.floor(F.datediff(F.current_date(), "DateOfBirth")/365.25).cast("int"))
    .withColumn("AgeBand", F.when(F.col("Age")<18,"0-17").when(F.col("Age")<40,"18-39")
        .when(F.col("Age")<65,"40-64").otherwise("65+")))
dim_department = bronze("dimdepartment").where(F.col("_IsCurrent"))
dim_facility   = bronze("dimfacility").where(F.col("_IsCurrent"))
dim_payer      = bronze("dimpayer").where(F.col("_IsCurrent"))
dim_diagnosis  = bronze("dimdiagnosis").where(F.col("_IsCurrent"))
dim_procedure  = bronze("dimprocedure").where(F.col("_IsCurrent"))
dim_provider_specialty  = bronze("dimproviderspecialty").where(F.col("_IsCurrent"))
dim_provider_credential = bronze("dimprovidercredential").where(F.col("_IsCurrent"))
bridge_provider_dept    = bronze("providerdepartmentbridge").where(F.col("_IsCurrent"))

# ---------- Facts: derive + enforce referential integrity ----------
valid_patient  = dim_patient.select("PatientKey")
valid_provider = dim_provider.select("ProviderKey")

fact_encounter = (bronze("factencounter")
    .withColumn("EncounterYear", F.year("EncounterDate"))
    .withColumn("EncounterMonth", F.date_format("EncounterDate","yyyy-MM"))
    .join(valid_patient, "PatientKey", "left_semi"))

fact_claim = (bronze("factclaim")
    .withColumn("DeniedFlag", (F.col("ClaimStatus")=="Denied"))
    .withColumn("AllowedVariance", F.col("BilledAmount")-F.col("AllowedAmount"))
    .withColumn("PatientResponsibility", F.col("AllowedAmount")-F.col("PaidAmount"))
    .withColumn("ServiceYear", F.year("ServiceDate"))
    .withColumn("ServiceMonth", F.date_format("ServiceDate","yyyy-MM"))
    .join(valid_patient, "PatientKey", "left_semi"))

fact_risk = bronze("factriskscore").join(valid_patient, "PatientKey", "left_semi")

silver = {
  "dim_provider":dim_provider, "dim_patient":dim_patient, "dim_department":dim_department,
  "dim_facility":dim_facility, "dim_payer":dim_payer, "dim_diagnosis":dim_diagnosis,
  "dim_procedure":dim_procedure, "dim_provider_specialty":dim_provider_specialty,
  "dim_provider_credential":dim_provider_credential, "bridge_provider_dept":bridge_provider_dept,
  "fact_encounter":fact_encounter, "fact_claim":fact_claim, "fact_risk_score":fact_risk,
}
for name, df in silver.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{SLV}{name}")
    print(f"{SLV+name:35s} {df.count():>8,} rows")
print("SILVER complete.")